# Neo4j Test Notebook

This notebook tests the main functionalities of the Neo4j database running in a Docker container.

First, you need to install the `neo4j` Python driver. You can do this by running the following command in your terminal:

In [1]:
!pip install neo4j

  Using cached pytz-2025.2-py2.py3-none-any.whl.metadata (22 kB)
Using cached pytz-2025.2-py2.py3-none-any.whl (509 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [neo4j]32m1/2 [neo4j]

[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: pip install --upgrade pip


Now, we can connect to the Neo4j database.

In [2]:
from neo4j import GraphDatabase

URI = "neo4j://localhost:7687"
AUTH = ("neo4j", "password")

with GraphDatabase.driver(URI, auth=AUTH) as driver:
    driver.verify_connectivity()

Let's create a node.

In [3]:
def create_person(tx, name):
    tx.run("CREATE (:Person {name: $name})", name=name)

with GraphDatabase.driver(URI, auth=AUTH) as driver:
    with driver.session() as session:
        session.execute_write(create_person, "Alice")

Now, let's create a relationship.

In [4]:
def create_relationship(tx, name1, name2):
    tx.run("MATCH (a:Person {name: $name1}) "
           "MATCH (b:Person {name: $name2}) "
           "MERGE (a)-[:KNOWS]->(b)", name1=name1, name2=name2)

with GraphDatabase.driver(URI, auth=AUTH) as driver:
    with driver.session() as session:
        session.execute_write(create_person, "Bob")
        session.execute_write(create_relationship, "Alice", "Bob")

Finally, let's query the data.

In [5]:
def find_person(tx, name):
    result = tx.run("MATCH (a:Person {name: $name}) RETURN a.name AS name", name=name)
    return [record["name"] for record in result]

with GraphDatabase.driver(URI, auth=AUTH) as driver:
    with driver.session() as session:
        names = session.execute_read(find_person, "Alice")
        for name in names:
            print(name)

Alice


Let's query the relationship

In [6]:
def find_relationship(tx):
    result = tx.run("MATCH (a:Person)-[r:KNOWS]->(b:Person) RETURN a.name, type(r), b.name")
    return [record for record in result]

with GraphDatabase.driver(URI, auth=AUTH) as driver:
    with driver.session() as session:
        relationships = session.execute_read(find_relationship)
        for rel in relationships:
            print(rel)

<Record a.name='Alice' type(r)='KNOWS' b.name='Bob'>
